In [7]:
import torch
import torch.nn as nn
import numpy as np

# LSTM

Advantages
* Solve gradient vanishing/exploding problem
* long term memory

Structure: hidden state (short term), cell state(long term)
* Forget gate: decide how much history to keep
* Input gate: how much new thing to write 
* Candidate memory: new thing
* Update long term memory: C_t = f_t * C_t-1 + i_t * C_t
* Output hidden = o_t * tanh(C_t)

### LSTM Network Equations

$$f_t = \sigma\left(W_f \cdot [h_{t-1}, x_t] + b_f\right)$$

$$i_t = \sigma\left(W_i \cdot [h_{t-1}, x_t] + b_i\right)$$

$$\tilde{C}_t = \tanh\left(W_C \cdot [h_{t-1}, x_t] + b_C\right)$$

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

$$o_t = \sigma\left(W_o \cdot [h_{t-1}, x_t] + b_o\right)$$

$$h_t = o_t \odot \tanh(C_t)$$


In [8]:
class LSTMClassifier(nn.Module):
    def __init__(self,
                 input_size: int,
                 hidden_size: int,
                 num_layers: int,
                 num_classes: int,
                 dropout: float = 0.2,
                 bidirectional: bool = False):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        self.fc = nn.Linear(hidden_size * self.num_directions, num_classes)

    def forward(self, x):
        # x: (batch_size, seq_len, input_size)

        output, (h_n, c_n) = self.lstm(x)

        # output: (batch_size, seq_len, hidden_size * num_directions)
        # h_n: (num_layers * num_directions, batch_size, hidden_size)
        # c_n: (num_layers * num_directions, batch_size, hidden_size)

        if self.bidirectional:
            # Last layer forward and backward hidden states
            forward_last = h_n[-2]
            backward_last = h_n[-1]
            last_hidden = torch.cat([forward_last, backward_last], dim=1)
        else:
            last_hidden = h_n[-1]

        logits = self.fc(last_hidden)
        return logits
        

In [12]:
x_t = torch.tensor([[1.0, 2.0]])          # (1, 2)
h_prev = torch.tensor([[0.1, 0.2, 0.3]]) # (1, 3)
c_prev = torch.tensor([[0.5, 0.5, 0.5]]) # (1, 3)

combined = torch.cat([x_t, h_prev], dim=1)
print(combined)

tensor([[1.0000, 2.0000, 0.1000, 0.2000, 0.3000]])


In [21]:
xh_to_gates = nn.Linear(5,3 * 4)
gates = xh_to_gates(combined)
print(gates)

tensor([[-1.1299,  0.5120, -0.0876, -0.1529, -0.6280,  0.8100,  1.2177, -0.2097,
          0.1079, -0.6385, -0.0494, -0.0320]], grad_fn=<AddmmBackward0>)


In [22]:
f_t, i_t, c_memo,o_t = torch.chunk(gates,chunks = 4,dim = 1)

In [23]:
f_t = torch.sigmoid(f_t)
i_t = torch.sigmoid(i_t)
c_memo = torch.tanh(c_memo)
o_t = torch.sigmoid(o_t)
c_t = f_t * c_prev + i_t * c_memo
h_t = o_t * torch.tanh(c_t)

In [24]:
h_t

tensor([[0.1623, 0.1152, 0.1493]], grad_fn=<MulBackward0>)

In [ ]:
class LSTMCellFromScratch(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size

        # One linear layer computes all 4 gates together:
        # forget, input, candidate, output
        self.xh_to_gates = nn.Linear(
            input_size + hidden_size,
            4 * hidden_size
        )

    def forward(self, x_t, h_prev, c_prev):
        # x_t:    (batch_size, input_size)
        # h_prev: (batch_size, hidden_size)
        # c_prev: (batch_size, hidden_size)

        combined = torch.cat([x_t, h_prev], dim=1)

        gates = self.xh_to_gates(combined)
        # truncate to 4 pieces
        f_t, i_t, c_tilde, o_t = torch.chunk(gates, chunks=4, dim=1)

        f_t = torch.sigmoid(f_t)          # forget gate
        i_t = torch.sigmoid(i_t)          # input gate
        c_tilde = torch.tanh(c_tilde)     # candidate memory
        o_t = torch.sigmoid(o_t)          # output gate

        c_t = f_t * c_prev + i_t * c_tilde
        h_t = o_t * torch.tanh(c_t)

        return h_t, c_t